First import catbird (ensure you've pip installed it and enabled the kernel in this notebook!)

In [1]:
from catbird import *

Some models to get started has been defined in the `models` directory. We'll import the simple breeder model

In [2]:
from models.simple_breeder import *

We need to define the path to a MOOSE executable. For convenience, one is provided in the catbird/app directory. Ensure it's compiled (`cd ../app/ && make`)

In [3]:
# Path to executable to obtain available syntax
app_exe="../app/dummy-opt"

We must create a factory to "pythonize" MOOSE syntax into python constructors. This step can take a few minutes. 
(NB in future we will look at cacheing data to speed up this step.)

In [4]:
# Custom factory of enabled MOOSE syntax
factory=SimpleBreederFactory(app_exe)

Loading syntax from library...
Done
Constructing syntax registry...
Done
Configuring objects to enable...
Done
Loading enabled objects...
Done


Initialise custom data structure to bundle default inputs

In [5]:
inputs=SimpleBreederInputs()

We could edit the default values:

In [6]:
# Update some material properties
# Value for Eurofer steel: Esteban et al 2007
inputs.materials["steel"].D0=4.57e-7 # m^2/2 J/mol
inputs.materials["steel"].E_d=22300 # J/(mol K)

# Values for LiAlO2: Roy et al 2024 
inputs.materials["breeder"].D0=1.56e-08  # m^2/2 J/mol
inputs.materials["breeder"].E_d=45680 # J/(mol K)

Now initialise our boilerplate model

In [7]:
model=SimpleBreederModel(factory,inputs)

Warning! Syntax collision for attribute control_tags in class Variables.Variable. Skipping.
Warning! Syntax collision for attribute family in class Variables.Variable. Skipping.
Warning! Syntax collision for attribute order in class Variables.Variable. Skipping.
Warning! Syntax collision for attribute scaling in class Variables.Variable. Skipping.
Warning! Syntax collision for attribute type in class Variables.Variable. Skipping.
Warning! Syntax collision for attribute control_tags in class AuxVariables.AuxVariable. Skipping.
Warning! Syntax collision for attribute family in class AuxVariables.AuxVariable. Skipping.
Warning! Syntax collision for attribute order in class AuxVariables.AuxVariable. Skipping.
Warning! Syntax collision for attribute scaling in class AuxVariables.AuxVariable. Skipping.
Warning! Syntax collision for attribute type in class AuxVariables.AuxVariable. Skipping.


 Let's inspect what blocks were created in our MOOSE model. This can be found in the convenience attribute `moose_objects`. Let's print it out:

In [8]:
print(model.moose_objects)

['executioner', 'problem', 'mesh', 'variables', 'auxvariables', 'ics', 'functions', 'kernels', 'materials', 'bcs', 'postprocessors', 'outputs']


 Let's inspect the Executioner.

In [9]:
help(model.executioner)

Help on Executioner in module abc object:

class Executioner(Executioner.Steady, catbird.action.MooseAction, catbird.collection.MooseCollection)
 |  Executioner(*args, **kwargs)
 |  
 |  MOOSE Object Parameters
 |  -----------------------
 |  accept_on_max_fixed_point_iteration : bool
 |    True to treat reaching the maximum number of fixed point iterations as converged.
 |    Default value: False
 |    Required: False
 |  
 |  accept_on_max_picard_iteration : bool
 |    True to treat reaching the maximum number of Picard iterations as converged.
 |    Default value: False
 |    Required: False
 |  
 |  auto_advance : bool
 |    Whether to automatically advance sub-applications regardless of whether their solve converges, for transient executioners only.
 |    Required: False
 |  
 |  automatic_scaling : bool
 |    Whether to use automatic scaling for the variables.
 |    Required: False
 |  
 |  compute_scaling_once : bool
 |    Whether the scaling factors should only be computed once

We can also programmatically obtain the same information using some convenience attributes:

In [10]:
print(model.executioner.type)
print(model.executioner.moose_object_params)

Steady
['accept_on_max_fixed_point_iteration', 'accept_on_max_picard_iteration', 'auto_advance', 'automatic_scaling', 'compute_scaling_once', 'contact_line_search_allowed_lambda_cuts', 'contact_line_search_ltol', 'control_tags', 'custom_abs_tol', 'custom_pp', 'custom_rel_tol', 'direct_pp_value', 'disable_fixed_point_residual_norm_check', 'disable_picard_residual_norm_check', 'enable', 'fixed_point_abs_tol', 'fixed_point_algorithm', 'fixed_point_force_norms', 'fixed_point_max_its', 'fixed_point_min_its', 'fixed_point_rel_tol', 'ignore_variables_for_autoscaling', 'l_abs_tol', 'l_max_its', 'l_tol', 'line_search', 'line_search_package', 'max_xfem_update', 'mffd_type', 'multi_system_fixed_point', 'multi_system_fixed_point_convergence', 'n_max_nonlinear_pingpong', 'nl_abs_div_tol', 'nl_abs_step_tol', 'nl_abs_tol', 'nl_div_tol', 'nl_forced_its', 'nl_max_funcs', 'nl_max_its', 'nl_rel_step_tol', 'nl_rel_tol', 'nonlinear_convergence', 'num_grids', 'off_diagonals_in_auto_scaling', 'outputs', 'pet

When setting attributes, catbird will first attempt to cast it as the expected type, and will raise an error if it's not possible.

In [11]:
model.executioner.fixed_point_abs_tol="Hello"

ValueError: Attribute fixed_point_abs_tol should have type <class 'float'>

Let's look at the materials:

This `Materials` class is just an instance of a catbird container called a `MooseCollection`.
We can obtain its length:

In [12]:
print(len(model.materials))

2


catbird's `MooseCollection` entries are stored in the `objects` dictionary. 

In [13]:
for mat_name in model.materials.objects.keys():
    print(mat_name)

D_steel
D_breeder


In [14]:
steel_diffusivity = model.materials.objects["D_steel"]

In [15]:
help(steel_diffusivity)

Help on Materials.Material in module abc object:

class Materials.Material(Materials.ADParsedMaterial, catbird.action.MooseAction)
 |  Materials.Material(*args, **kwargs)
 |  
 |  MOOSE Object Parameters
 |  -----------------------
 |  block : str
 |    The list of blocks (ids or names) that this object will be applied
 |    Required: False
 |  
 |  boundary : str
 |    The list of boundaries (ids or names) from the mesh where this object applies
 |    Required: False
 |  
 |  compute : bool
 |    When false, MOOSE will not call compute methods on this material. The user must call computeProperties() after retrieving the MaterialBase via MaterialBasePropertyInterface::getMaterialBase(). Non-computed MaterialBases are not sorted for dependencies.
 |    Default value: True
 |    Required: False
 |  
 |  constant_expressions : str
 |    Vector of values for the constants in constant_names (can be an FParser expression)
 |    Required: False
 |  
 |  constant_names : str
 |    Vector of co

Let's inspect the some of attributes' values:

In [16]:
print(steel_diffusivity.expression)
print(steel_diffusivity.coupled_variables)
print(steel_diffusivity.constant_names)
print(steel_diffusivity.constant_expressions)

D0 * exp(-E_d / (R * temperature))
temperature
D0 E_d R
4.57e-07 22300 8.31


We could change these parameter variables - maybe we want to do some UQ :-)
Note MOOSE expects a string for the attribute `constant_expressions`

In [17]:
print(type(steel_diffusivity.constant_expressions))
my_D0=2e-7
my_Ed=25000
steel_diffusivity.constant_expressions="{} {} {} ".format(my_D0,my_Ed,R_univ)
print(steel_diffusivity.constant_expressions)

<class 'str'>
2e-07 25000 8.31 


When we've finished modifing the boiler-plate, we can write to file.

In [18]:
input_name="simple_breeder.i"
model.write(input_name)

Wrote to  simple_breeder.i


Let's look at what we wrote:

In [19]:
!cat simple_breeder.i

[Executioner]
  type=Steady
  line_search=none
  petsc_options_iname='-pc_type -pc_factor_mat_solver_package'
  petsc_options_value='lu superlu_dist'
  solve_type=NEWTON
[]
[Problem]
  type=FEProblem
[]
[Mesh]
  [polygon_rings]
    type=PolygonConcentricCircleMeshGenerator
    background_block_names=outer_void
    create_inward_interface_boundaries=True
    external_boundary_name=outer
    flat_side_up=True
    interface_boundary_id_shift=1000
    inward_interface_boundary_names='inner_steel_inner_void breeder_inner_steel outer_steel_breeder outer_void_outer_steel'
    num_sectors_per_side='10 10 10 10'
    num_sides=4
    outward_interface_boundary_names='inner_void_inner_steel inner_steel_breeder breeder_outer_steel outer_steel_outer_void'
    polygon_size=0.088
    ring_block_names='inner_void inner_steel breeder outer_steel'
    ring_intervals='1 8 8 8'
    ring_radii='0.012 0.02 0.028 0.036000000000000004'
  []
  [delete_inner]
    type=BlockDeletionGenerator
    block=inner_void


All is looking good! Let's run the input.

In [20]:
!../app/dummy-opt -i simple_breeder.i

In UnstructuredMesh::stitch_meshes:
This mesh has 27 nodes on boundary `' (30000).
Other mesh has 27 nodes on boundary `' (31000).
Minimum edge length on both surfaces is 0.00100231.
In UnstructuredMesh::stitch_meshes:
Found 27 matching nodes.

In UnstructuredMesh::stitch_meshes:
This mesh has 27 nodes on boundary `' (30000).
Other mesh has 27 nodes on boundary `' (31000).
Minimum edge length on both surfaces is 0.00100231.
In UnstructuredMesh::stitch_meshes:
Found 27 matching nodes.

In UnstructuredMesh::stitch_meshes:
This mesh has 27 nodes on boundary `' (30000).
Other mesh has 27 nodes on boundary `' (31000).
Minimum edge length on both surfaces is 0.00100231.
In UnstructuredMesh::stitch_meshes:
Found 27 matching nodes.

In UnstructuredMesh::stitch_meshes:
This mesh has 27 nodes on boundary `' (30000).
Other mesh has 27 nodes on boundary `' (31000).
Minimum edge length on both surfaces is 0.00100231.
In UnstructuredMesh::stitch_meshes:
Found 27 matching nodes.

                    

We should have written two files, let's check they exist.

In [21]:
!ls simple_breeder_out.*

simple_breeder_out.csv	simple_breeder_out.e


Postprocessors were saved to csv file so we can easily retrieve the data.

In [22]:
!cat simple_breeder_out.csv

time,diffusive_flux_integral
0,0
1,1.0546611336364e-09


Now it's over to the analyst to decide how to utilise the model. Perhaps change some variables and make a plot!